## 熵权法 + TOPSIS（Python 3.11）

**一、熵权法：让数据自己"投票"决定权重**

1. **正向化**：把极小型、中间型、区间型指标统一转成"越大越好"
2. **标准化**：消除量纲 $z_{ij}=\dfrac{x_{ij}}{\sqrt{\sum_i x_{ij}^2}}$
3. **计算比重**：$p_{ij}=\dfrac{z_{ij}}{\sum_i z_{ij}}$
4. **信息熵**：$e_j=-\dfrac{1}{\ln n}\sum_i p_{ij}\ln p_{ij}$（约定 $0\ln 0=0$）
5. **熵权**：$w_j=\dfrac{1-e_j}{\sum_k (1-e_k)}$ —— 指标差异越大，信息越多，权重越大

**二、TOPSIS：离正理想解越近、离负理想解越远越好**

1. 加权规范化矩阵 $v_{ij}=w_j z_{ij}$
2. 正理想解 $v_j^+=\max_i v_{ij}$，负理想解 $v_j^-=\min_i v_{ij}$
3. 欧氏距离 $D_i^{\pm}=\sqrt{\sum_j (v_{ij}-v_j^{\pm})^2}$
4. 贴近度 $C_i=\dfrac{D_i^-}{D_i^+ + D_i^-}\in[0,1]$，越大越优

**三、指标正向化公式**

| 类型 | 含义 | 正向化公式 |
| --- | --- | --- |
| `benefit` | 极大型（效益型） | $y=x$ |
| `cost` | 极小型（成本型） | $y=\max(x)-x$ |
| `intermediate` | 中间型（最优值 $x_{best}$） | $y=1-\dfrac{|x-x_{best}|}{\max|x-x_{best}|}$ |
| `interval` | 区间型（最优区间 $[a,b]$） | $y=1-\dfrac{d(x,[a,b])}{\max d(x,[a,b])}$ |


### 1. 环境准备

In [ ]:
# 熵权法 + TOPSIS（Python 3.11）
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 中文字体，避免图表中文乱码
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 120)

print("Python 版本：", sys.version.split()[0])


### 2. 指标正向化

In [ ]:
KINDS = ("benefit", "cost", "intermediate", "interval")


def positivize(X, kinds, params=None):
    """把所有指标统一正向化为"越大越好"。

    X      : (n_samples, m_indicators) 原始数据
    kinds  : 长度 m 的列表，元素取自 KINDS
    params : 与 kinds 一一对应的参数
             intermediate -> x_best（最优值）
             interval     -> (a, b)（最优区间）
    返回正向化后的矩阵（与 X 同形状）
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X 必须是二维矩阵（样本 × 指标）")
    if len(kinds) != X.shape[1]:
        raise ValueError("kinds 长度必须等于指标个数")
    if not set(kinds) <= set(KINDS):
        raise ValueError(f"指标类型只能是 {KINDS}")

    params = list(params) if params is not None else [None] * len(kinds)
    if len(params) != len(kinds):
        raise ValueError("params 长度必须等于指标个数")

    Y = X.copy()
    col_max = X.max(axis=0)

    for j, kind in enumerate(kinds):
        col = X[:, j]
        if kind == "benefit":
            Y[:, j] = col
        elif kind == "cost":
            Y[:, j] = col_max[j] - col
        elif kind == "intermediate":
            x_best = float(params[j])
            gap = np.abs(col - x_best)
            top = gap.max()
            Y[:, j] = 1.0 - gap / top if top > 0 else 1.0
        else:  # interval
            a, b = params[j]
            lo, hi = min(a, b), max(a, b)
            below = np.where(col < lo, lo - col, 0.0)   # 低于下界的距离
            above = np.where(col > hi, col - hi, 0.0)   # 高于上界的距离
            dist = below + above
            top = dist.max()
            Y[:, j] = 1.0 - dist / top if top > 0 else 1.0

    return Y


### 3. 熵权法求权重

In [ ]:
def entropy_weight(X):
    """熵权法求权重，要求 X 已正向化且非负。

    返回 (w, Z, e)：
      w —— 各指标权重，和为 1
      Z —— 向量归一化后的矩阵，供 TOPSIS 使用
      e —— 各指标信息熵（越接近 1 说明差异越小、权重越低）
    """
    X = np.asarray(X, dtype=float)
    if X.ndim != 2:
        raise ValueError("X 必须是二维矩阵")
    if (X < 0).any():
        raise ValueError("熵权法要求数据非负，请先用 positivize 正向化")
    n, m = X.shape
    if n < 2:
        raise ValueError("样本数至少为 2")

    # 1. 向量归一化，消除量纲
    norm = np.linalg.norm(X, axis=0)
    Z = X / np.where(norm == 0, 1.0, norm)

    # 2. 计算第 j 个指标下第 i 个样本的比重
    col_sum = Z.sum(axis=0)
    P = Z / np.where(col_sum == 0, 1.0, col_sum)

    # 3. 信息熵（约定 0 * ln0 = 0）
    with np.errstate(divide="ignore", invalid="ignore"):
        L = np.where(P > 0, P * np.log(P), 0.0)
    e = -L.sum(axis=0) / np.log(n)

    # 4. 差异系数归一化得到权重；若各列完全相同则退化为等权
    d = 1.0 - e
    total = d.sum()
    w = d / total if total > 0 else np.full(m, 1.0 / m)
    return w, Z, e


### 4. TOPSIS 综合评价

In [ ]:
def topsis(Z, w):
    """加权 TOPSIS，输入为标准化矩阵 Z 与权重 w。

    返回 (C, d_pos, d_neg, v_best, v_worst)：
      C —— 贴近度，取值 [0, 1]，越大越优
    """
    Z = np.asarray(Z, dtype=float)
    w = np.asarray(w, dtype=float)
    if Z.ndim != 2 or Z.shape[1] != w.size:
        raise ValueError("Z 的列数必须与权重个数一致")

    V = Z * w                                       # 加权规范化矩阵
    v_best = V.max(axis=0)                          # 正理想解
    v_worst = V.min(axis=0)                         # 负理想解
    d_pos = np.sqrt(((V - v_best) ** 2).sum(axis=1))
    d_neg = np.sqrt(((V - v_worst) ** 2).sum(axis=1))

    denom = d_pos + d_neg
    C = np.divide(d_neg, denom, out=np.zeros_like(d_neg), where=denom > 0)

    # 处理正负理想解重合（所有方案完全相同）的退化情形
    C = np.where(denom > 0, C, 1.0)
    return C, d_pos, d_neg, v_best, v_worst


def ewm_topsis(X, kinds, params=None, names=None, index=None, show=True):
    """熵权法 + TOPSIS 一体化评价，返回结果 DataFrame（按贴近度降序）。"""
    X = np.asarray(X, dtype=float)
    m = X.shape[1]
    names = list(names) if names is not None else [f"指标{j + 1}" for j in range(m)]

    Y = positivize(X, kinds, params)
    w, Z, e = entropy_weight(Y)
    C, d_pos, d_neg, _, _ = topsis(Z, w)

    if show:
        table = pd.DataFrame({
            "指标": names,
            "类型": kinds,
            "信息熵 e": e,
            "差异系数 1-e": 1.0 - e,
            "权重 w": w,
        }).set_index("指标")
        print("=== 熵权法权重 ===\n")
        print(table.round(4).to_string())

    result = pd.DataFrame({
        "贴近度 C": C,
        "正理想解距离": d_pos,
        "负理想解距离": d_neg,
    }, index=index)
    result["排名"] = result["贴近度 C"].rank(ascending=False, method="min").astype(int)
    return result.sort_values("贴近度 C", ascending=False)


### 5. 示例：5 个城市的宜居性评价

指标（均为常见公开统计数据量级的模拟值）：

| 指标 | 类型 | 说明 |
| --- | --- | --- |
| 人均 GDP（万元） | 极大型 | 越高越好 |
| 单位 GDP 能耗（吨标煤/万元） | 极小型 | 越低越好 |
| 年平均气温（℃） | 中间型 | 最优 22℃ |
| 绿化覆盖率（%） | 区间型 | 最优 40%~50% |
| 教育投入占比（%） | 极大型 | 越高越好 |


In [ ]:
cities = ["北京", "上海", "广州", "成都", "拉萨"]
indicators = ["人均GDP", "单位GDP能耗", "年平均气温", "绿化覆盖率", "教育投入占比"]

X = np.array([
    [20.0, 0.30, 12.9, 48.5, 5.6],
    [18.5, 0.35, 16.6, 39.6, 4.8],
    [16.0, 0.40, 22.4, 43.2, 4.5],
    [10.5, 0.45, 16.2, 44.9, 5.1],
    [7.2,  0.25,  8.5, 38.0, 6.2],
], dtype=float)

kinds = ["benefit", "cost", "intermediate", "interval", "benefit"]
params = [None, None, 22.0, (40.0, 50.0), None]

df = pd.DataFrame(X, index=cities, columns=indicators)
df


In [ ]:
Y = positivize(X, kinds, params)
pd.DataFrame(Y.round(4), index=cities, columns=indicators)


In [ ]:
result = ewm_topsis(X, kinds, params, names=indicators, index=cities)
print("\n=== 综合评价结果（按贴近度降序）===\n")
result.round(4)


### 6. 结果可视化

In [ ]:
w, Z, e = entropy_weight(Y)   # 复用上面已正向化的 Y

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 左图：熵权
order = np.argsort(w)
axes[0].barh(np.array(indicators)[order], w[order], color="#4C72B0")
axes[0].set_title("熵权法权重")
axes[0].set_xlabel("权重")
for k, idx in enumerate(order):
    axes[0].text(w[idx], k, f" {w[idx]:.3f}", va="center", fontsize=9)

# 右图：贴近度排名
score = result["贴近度 C"]
axes[1].bar(score.index, score.values, color="#55A868")
axes[1].set_title("TOPSIS 贴近度排名")
axes[1].set_ylabel("贴近度 C")
axes[1].set_ylim(0, 1)
for name, value in score.items():
    axes[1].text(name, value, f"{value:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()
